# Dataset loading and synchronisation example

In this example we will analyse data from a toy experiment where a user must speed-type based on a series of word prompts.

In [31]:
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import swc.aeon.io.api as aeon
from dotmap import DotMap
from swc.aeon.io.video import frames
from swc.aeon.schema.streams import Device

from ucl_open.io.streams import csv, harp_device, raw_video

## Define the experiment

Specify a root folder and data mapping

In [32]:
root = Path(r"C:\temp_data\sub-Andrew\ses-001_date-2026-10-05T14-41-31")

experiment = DotMap(
    [
        Device("Video", raw_video()),
        Device("TrialResult", csv(columns=("subject", "target"))),
        Device("KeyInput", csv(columns=["key"])),
        Device("CurrentWord", csv(columns=["current_word"])),
        Device("AsyncPulse", csv(columns=["pulse"])),
    ]
)

print(experiment.keys())

odict_keys(['Video', 'TrialResult', 'KeyInput', 'CurrentWord', 'AsyncPulse'])


## Load the dataset

In [34]:
video = aeon.load(root, experiment.Video)
trial_result = aeon.load(root, experiment.TrialResult)  
key_input = aeon.load(root, experiment.KeyInput)
current_word = aeon.load(root, experiment.CurrentWord)
async_pulse = aeon.load(root, experiment.AsyncPulse)
video.head()

,width,height,_frame,_path,_epoch
time,,,,,
1904-01-01 00:00:00.297000+00:00,640,480,0,C:\temp_data\sub-Andrew\ses-001_date-2026-10-0...,ses-001_date-2026-10-05T14-41-31
1904-01-01 00:00:00.319000+00:00,640,480,1,C:\temp_data\sub-Andrew\ses-001_date-2026-10-0...,ses-001_date-2026-10-05T14-41-31
1904-01-01 00:00:00.341000+00:00,640,480,2,C:\temp_data\sub-Andrew\ses-001_date-2026-10-0...,ses-001_date-2026-10-05T14-41-31
1904-01-01 00:00:00.363000+00:00,640,480,3,C:\temp_data\sub-Andrew\ses-001_date-2026-10-0...,ses-001_date-2026-10-05T14-41-31
1904-01-01 00:00:00.385000+00:00,640,480,4,C:\temp_data\sub-Andrew\ses-001_date-2026-10-0...,ses-001_date-2026-10-05T14-41-31


## Synchronisation check

This dataset has been deliberately collected with two separate timebases with slightly drifting clocks. User input and trial events (key presses, word generation etc.) are on one clock; and video data is collected on a different clock. We have added an example async pulse link in which a sample from the video data clock is periodically sent to a device on the user input clock. These samples are logged under `AsyncPulse`, where the time index is the user index clock and the value is the corresponding time from the video data clock.

We can observe the synchronisation problem by inspecting video frames where the `Enter` key is pressed. 